## **MLlib 튜토리얼**

In [ ]:
# [+] SparkSession 설정
...

### **MLlib 예제: Logistic Regression**
+ ```pyspark.ml.linalg```: 선형대수(linear algebra) 관련 패키지
    + ```Vectors```: 벡터 데이터 타입
+ ```pyspark.ml.classification```: 분류 모델 관련 패키지
    + ```LogisticRegression```: 가장 기본적인 분류 모델

In [ ]:
# Vectors, LogisticRegression 임포트
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import LogisticRegression

**데이터 정의: 레이블과 특징**
+ ```dense()```: 모든 값을 저장하는 벡터 생성
+ ```sparse()```: 0이 아닌 값만 저장하는 벡터 생성

In [ ]:
# 훈련 데이터: (label, features)
train_data = [
    (1.0, Vectors.dense([0.0, 1.1, 0.1])),
    (0.0, Vectors.dense([2.0, 1.0, -1.0])),
    (0.0, Vectors.dense([2.0, 1.3, 1.0])),
    (1.0, Vectors.dense([0.0, 1.2, -0.5])),
]

In [ ]:
# [+] 스키마 정의
schema = ...

In [ ]:
# [+] 데이터프레임 생성
train_df = ...

In [ ]:
# 데이터프레임 출력 
train_df.show()

**LogisticRegression (Estimator) 생성**
+ ```maxIter```: 최대 학습 횟수 매개변수
+ ```regParam```: 정규화 매개변수

In [1]:
# [+] Estimator(학습 알고리즘) 객체 생성
lr = ...

In [ ]:
# [+] 모델 학습
model = ...

In [ ]:
# 모델 출력
model

In [ ]:
# 테스트셋
test_data = [
    (1.0, Vectors.dense([-1.0, 1.5, 1.3])),
    (0.0, Vectors.dense([3.0, 2.0, -0.1])),
    (1.0, Vectors.dense([0.0, 2.2, -1.5])),
]

In [ ]:
# [+] 데이터프레임 생성
test_df = ...

In [ ]:
# [+] 모델 예측
predictions = ...

In [ ]:
# 예측결과 출력
predictions.show()

### **LogisticRegression 파이프라인 구현하기**
+ 예측 문제: Spark에 대한 텍스트 인지 아닌지를 분류
+ ```Pipeline```: 머신러닝 과정에 대한 파이프라인 정의
+ ```HashingTF```: 컬럼을 용어빈도(term-frequency) 컬럼으로 변환하는 Transformer
+ ```Tokenizer```: 컬럼의 텍스트를 여러 개의 단어로 분할하는 Transformer

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, Tokenizer

In [ ]:
# 훈련 데이터를 데이터프레임으로 생성
train_df = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0)
], ["id", "text", "label"])

In [ ]:
train_df.show()

**머신러닝 수행 과정**

Tokenizer ${\rightarrow}$ HashingTF ${\rightarrow}$ LogisticRegression

In [ ]:
# [+] Tokenizer 객체 생성
tokenizer = ...

In [ ]:
# 토크나이징 수행
df_tok = tokenizer.transform(train_df)
df_tok.show()

In [ ]:
# [+] HashingTF 객체 생성
hashingTF = ...

In [ ]:
# 단어빈도 벡터로 변환
df_tf = hashingTF.transform(df_tok)
df_tf.show()

**LogisticRegression 객체 생성**
+ ```maxIter```: 10
+ ```regParam```: 0.001

In [ ]:
# [+] Estimator 객체 생성
lr = ...

In [ ]:
# 모델 학습
model = lr.fit(df_tf)

In [ ]:
# 모델 학습 결과(특징별 계수인터셉트)
print(model.coefficients)
print(model.intercept)

**머신러닝 파이프라인 생성 및 실행**\
```Pipeline.fit()```은 다음의 절차들을 일괄적으로 수행
1. 토크나이징(Tokenizer): text ${\rightarrow}$ words
2. 단어빈도(Term-Frequency) 벡터로 변환: words ${\rightarrow}$ features
3. 로지스틱스 회귀모델(LogisticRegression) 학습

In [ ]:
# [+] Pipeline 객체 생성
pipeline = ...

In [ ]:
# [+] Pipeline 실행
model = ...

In [ ]:
# 테스트 데이터에 대한 DataFrame 생성
test_df = spark.createDataFrame([
    (4, "spark i j k"),
    (5, "l m n"),
    (6, "spark hadoop spark"),
    (7, "apache hadoop")
], ["id", "text"])

In [ ]:
# [+] 테스트 데이터에 대한 예측
predictions = ...

In [ ]:
predictions.show()